# P4: Imlo tuzatish va LSH qidiruv tizimi
**Mavzu:** Levenshtein tahrir masofasi, Noisy Channel imlo tuzatish, MinHash LSH qidiruv
**Kun:** 5-kun amaliyoti — 22-iyun 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L4 — Masofaga asoslangan qidiruv va imlo tuzatish
**Kapstone modul:** m04 — `SpellLSHRetriever` (`capstone/modules/m04_spell_lsh_retriever.py`)
**Ma'lumot:** uz_news_full (1000+ hujjat, LSH benchmark). Offline rejimda kichik
namuna (`d05_checkpoints/uz_news_corpus.txt`) ishlatiladi — datasketch va 240 MB shart emas.

**Bugungi maqsadlar:**
1. Levenshtein tahrir masofasini DP bilan implementatsiya qilish
2. Noisy Channel ($\arg\max P(w)\,P(x{\mid}w)$) bilan imlo tuzatish
3. MinHash LSH indeksini qurish
4. LSH va oddiy (k-NN) qidiruv tezligini taqqoslash
5. m04 `SpellLSHRetriever` ni `capstone/contracts.py` imzolariga muvofiq yozish

| Bo'lim | Vaqt |
|--------|------|
| §1 Muhit tekshiruvi | 3 daq |
| §2 Yaxlit natija | 8 daq |
| §3 Tayyor kod bloki — PRIMM | 17 daq |
| §4 Asosiy mavzu — so'nuvchi tayanch | 35 daq |
| §5 Kapstone integratsiya | 13 daq |
| §6 Tadqiqot + yakun | 7 daq |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# §1  Muhit tekshiruvi
# ═══════════════════════════════════════════════════════════════
import random, os, sys, time
from pathlib import Path
import numpy as np

random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True   # <-- bu qiymatni o'zgartirmang

CHECKPOINT_DIR = Path("d05_checkpoints")
assert CHECKPOINT_DIR.exists(), (
    "d05_checkpoints/ papkasi topilmadi. "
    "Notebookni course/practices/ papkasidan ishga tushiring yoki: git pull."
)

REPO_ROOT = Path().resolve().parent.parent
MODULES_DIR = REPO_ROOT / "capstone" / "modules"
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

try:
    import datasketch
    HAS_DATASKETCH = True
except ImportError:
    HAS_DATASKETCH = False

import sklearn
print(f"Python      : {sys.version.split()[0]}")
print(f"numpy       : {np.__version__}")
print(f"datasketch  : {'bor' if HAS_DATASKETCH else 'YO'+chr(39)+'Q (offline toza-python LSH ishlatiladi)'}")
print("\n✓ Muhit tayyor.")


---
## §2  Yaxlit Natija — Pirovard Manzil Birinchi! (~8 daqiqa)

Quyida tugallangan tizim demosi: xato so'zni \textbf{tuzatamiz} va so'rovga
\textbf{o'xshash hujjatlarni} topamiz. Avval natijani ko'ring.


In [ ]:
# Pirovard natija (inline versiya — m04 hali yozilmagan)
from pathlib import Path
from collections import Counter

CORPUS = [l.strip() for l in Path("d05_checkpoints/uz_news_corpus.txt").open(encoding="utf-8") if l.strip()]

def _ed(s1, s2):   # Levenshtein (qisqa inline)
    m, n = len(s1), len(s2)
    D = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): D[i][0] = i
    for j in range(n+1): D[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            D[i][j] = (D[i-1][j-1] if s1[i-1]==s2[j-1]
                       else 1+min(D[i-1][j], D[i][j-1], D[i-1][j-1]))
    return D[m][n]

freq = Counter(w for d in CORPUS for w in d.split())
def _correct(x):
    if x in freq: return x
    cand = [(freq[w]*(0.1**_ed(x, w)), w) for w in freq if _ed(x, w) <= 2]
    return max(cand)[1] if cand else x

print(f"Korpus: {len(CORPUS)} hujjat")
print(f"correct('telfon')   -> {_correct('telfon')}")
print(f"correct('internt')  -> {_correct('internt')}")
q = CORPUS[0]
print(f"\nSo'rov: '{q}'")
sim = sorted(CORPUS, key=lambda d: -len(set(d.split()) & set(q.split())))[:3]
print("Eng o'xshash hujjatlar:")
for d in sim: print(f"  - {d}")
print("\n✓ Tizim ishladi! Quyida har qadamni o'rganamiz.")


---
## §3  Tayyor Kod Bloki — PRIMM (~17 daqiqa)

### 3A. MinHash LSH g'oyasi (datasketch yoki toza-python)

> **Bashorat qiling:**
> Millionlab hujjat orasidan o'xshashini topish uchun har biriga solishtirish
> ($O(N)$) sekin. LSH o'xshash hujjatlarni bir "savat"ga tushiradi. Sizningcha,
> bu qanday qilib qidiruvni tezlashtiradi?


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
# Kaggle (onlayn) da sanoat kutubxonasi datasketch ishlatiladi:
#   from datasketch import MinHash, MinHashLSH
#   lsh = MinHashLSH(threshold=0.5, num_perm=64)
#   for i, doc in enumerate(CORPUS):
#       m = MinHash(num_perm=64)
#       for tok in set(doc.split()):
#           m.update(tok.encode("utf-8"))
#       lsh.insert(str(i), m)
#   candidates = lsh.query(query_minhash)   # faqat o'xshash savatdagilar
#
# Bu amaliyotda (offline) datasketch SHART EMAS — m04 ichida toza-python
# MinHash+LSH bor. Quyida uning g'oyasini ko'ramiz:
import zlib
def stable_hash(s):
    return zlib.crc32(s.encode("utf-8")) & 0x7FFFFFFF

# Bitta hujjat uchun MinHash "imzosi" (4 ta xesh, namuna uchun):
rng = np.random.RandomState(42)
a = rng.randint(1, 2**31-1, 4); b = rng.randint(0, 2**31-1, 4); P = 2**31-1
def minhash_sig(tokens):
    hs = np.array([stable_hash(t) for t in tokens])
    return [int(((a[k]*hs + b[k]) % P).min()) for k in range(4)]

print("MinHash imzo (2 o'xshash hujjat yaqin imzo beradi):")
print("  D0:", minhash_sig(set(CORPUS[0].split())))
print("  D1:", minhash_sig(set(CORPUS[1].split())))
print(f"\nHolat: datasketch = {HAS_DATASKETCH} -> {'sanoat kutubxonasi' if HAS_DATASKETCH else 'toza-python (m04)'}")


> **Tekshiring:**
> 1. Ikki hujjat ko'p umumiy so'zga ega bo'lsa, MinHash imzolari yaqinroqmi?
> 2. `stable_hash` nega oddiy `hash()` o'rniga ishlatilgan? (Maslahat: takrorlanuvchanlik.)
> 3. `num_perm` ni oshirsak, Jaccard bahosi aniqroq bo'ladimi?

> **O'zgartiring:** boshqa ikki hujjat (`CORPUS[2]`, `CORPUS[3]`) imzosini solishtiring.


### 3B. Unigram chastota lug'ati va $P(w)$

> **Bashorat qiling:**
> Noisy channel imlo tuzatish $P(w)$ (til modeli) ga muhtoj. $P(w)$ ni
> qayerdan olamiz? Korpusdagi so'z chastotasidan. Eng tez-tez uchraydigan so'z qaysi?


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
# m01 (P1 da qurgan modul) bilan tokenize qilamiz — kapstone uzviyligi
from m01_text_preprocessor import TextPreprocessor
pre = TextPreprocessor()

freq = Counter()
for doc in CORPUS:
    freq.update(pre.preprocess(doc))
total = sum(freq.values())

def P_word(w):
    return freq[w] / total

print(f"Lug'at hajmi : {len(freq)} unikal so'z")
print(f"Jami tokenlar: {total}")
print("Eng tez-tez uchraydigan 5 so'z:")
for w, c in freq.most_common(5):
    print(f"  {w:14s} count={c:3d}  P(w)={P_word(w):.4f}")


> **Tekshiring:**
> 1. Eng tez uchraydigan so'zlar kontent so'zlarmi yoki yordamchilar?
> 2. `P_word('telefon')` nechiga teng? Bu noisy channel da nima rol o'ynaydi?
> 3. m01 stemming lug'at hajmini qanday o'zgartirdi?


In [ ]:
# ─── CHECKPOINT A ─────────────────────────────────────────────────────────
# CORPUS, freq, total, pre keyingi bo'limlar uchun tayyor.
if "CORPUS" not in dir() or "freq" not in dir():
    from m01_text_preprocessor import TextPreprocessor
    pre = TextPreprocessor()
    CORPUS = [l.strip() for l in Path("d05_checkpoints/uz_news_corpus.txt").open(encoding="utf-8") if l.strip()]
    freq = Counter(w for d in CORPUS for w in pre.preprocess(d))
    total = sum(freq.values())
print(f"Checkpoint A: {len(CORPUS)} hujjat, {len(freq)} so'zli lug'at tayyor.")


---
## §4  Asosiy Mavzu — So'nuvchi Tayanch (~35 daqiqa)

**Namuna** (Men ko'rsataman) → **Birgalikda** (`# === SIZNING KODINGIZ ===`)
→ **Mustaqil** (Siz qilasiz).


### 4A. Namuna: Levenshtein Tahrir Masofasi — L4 [I3]-Slayd

DP jadval bilan minimal tahrir (qo'shish/o'chirish/almashtirish) sonini topamiz.
**Bu katak natijasi P4 birinchi assertini belgilaydi.**


In [ ]:
# ═══ NAMUNA (Men ko'rsataman): edit_distance DP ═══
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    D = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): D[i][0] = i      # o'chirish
    for j in range(n+1): D[0][j] = j      # qo'shish
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                D[i][j] = D[i-1][j-1]
            else:
                D[i][j] = 1 + min(D[i-1][j],      # o'chirish
                                  D[i][j-1],      # qo'shish
                                  D[i-1][j-1])    # almashtirish
    return D[m][n]

d_qol = edit_distance("qo'l", "ko'l")
d_das = edit_distance("dastur", "dastir")
print(f"edit_distance(qo'l, ko'l)      = {d_qol}")
print(f"edit_distance(dastur, dastir)  = {d_das}")

# ─── ASSERT: Ma'ruza L4 [I3]-slayd bilan solishtiring ─────────────────────
assert edit_distance("qo'l", "ko'l") == 1, "qo'l->ko'l 1 ta almashtirish (q->k)."
assert edit_distance("dastur", "dastir") == 1, "dastur->dastir 1 ta almashtirish (u->i)."
# Ma'ruza L4 [I3]-slayd bilan solishtiring
print("\n✓ Ma'ruza L4 [I3]-slayd tasdiqlandi: edit_distance = 1")


### 4B. Birgalikda: Noisy Channel imlo tuzatish

$\hat{w} = \arg\max_w P(w)\cdot P(x{\mid}w)$, bunda $P(x{\mid}w) = \alpha^{\text{edit}(x,w)}$
(yaqin so'z — ehtimoliyroq). Nomzodlar: tahrir masofasi $\le 2$ bo'lgan lug'at so'zlari.


In [ ]:
# ═══ BIRGALIKDA: correct() ═══
ALPHA = 0.1   # kanal modeli: P(x|w) = ALPHA ** edit_distance

def correct(x):
    x = x.lower()
    if x in freq:               # lug'atda bor -> tuzatish shart emas
        return x
    best, best_score = x, -1.0
    for w in freq:
        d = edit_distance(x, w)
        if d > 2:
            continue
        # === SIZNING KODINGIZ (1 qator) ===
        # noisy channel ball: P(w) * ALPHA**d   (P(w) = freq[w]/total)
        score = None
        if score is not None and score > best_score:
            best, best_score = w, score
    return best

print(f"correct('telfon')  -> {correct('telfon')}")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert correct("telfon") == "telefon", (
    f"correct('telfon') = {correct('telfon')!r}, kutilgan 'telefon'. "
    "Noisy channel ballini tekshiring: P(w)*ALPHA**d."
)
assert correct("internt") == "internet", "correct('internt') 'internet' bo'lishi kerak."
assert correct("telefon") == "telefon", "Lug'atdagi so'z o'zgarmasligi kerak."
print("✓ Noisy channel to'g'ri: telfon->telefon, internt->internet")


### 4C. Birgalikda: MinHash LSH indeks va k-NN bilan tezlik taqqoslash

LSH faqat bir savatga tushgan nomzodlarni tekshiradi (k-NN esa hammasini).
Avval LSH indeksini quramiz, keyin nomzodlar sonini $N$ bilan taqqoslaymiz.


In [ ]:
# ═══ BIRGALIKDA: MinHash LSH indeks ═══
NUM_PERM, BANDS = 64, 16
ROWS = NUM_PERM // BANDS
rng = np.random.RandomState(42)
A = rng.randint(1, 2**31-1, NUM_PERM); B = rng.randint(0, 2**31-1, NUM_PERM); PRIME = 2**31-1

def shingles(text):
    return set(pre.preprocess(text))

def signature(sh):
    if not sh:
        return tuple([PRIME]*NUM_PERM)
    hs = np.array([stable_hash(t) for t in sh])
    Msig = (A[:,None]*hs[None,:] + B[:,None]) % PRIME
    return tuple(int(x) for x in Msig.min(axis=1))

doc_sh = [shingles(d) for d in CORPUS]
buckets = {}
for i, sh in enumerate(doc_sh):
    sig = signature(sh)
    for band in range(BANDS):
        key = (band, sig[band*ROWS:(band+1)*ROWS])
        buckets.setdefault(key, set()).add(i)

def lsh_candidates(query):
    sig = signature(shingles(query))
    cand = set()
    for band in range(BANDS):
        key = (band, sig[band*ROWS:(band+1)*ROWS])
        # === SIZNING KODINGIZ (1 qator) ===
        # shu band kalitidagi hujjatlarni cand ga qo'shing (buckets.get(key, set()))
        pass
    return cand

query = CORPUS[0]
cand = lsh_candidates(query)
print(f"So'rov: '{query}'")
print(f"LSH nomzodlar soni: {len(cand)}  (jami hujjat: {len(CORPUS)})")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
def jaccard(p, q):
    return len(p & q) / len(p | q) if (p and q) else 0.0

# LSH qidiruv (faqat nomzodlar)
def retrieve_lsh(query, k=5):
    qsh = shingles(query)
    scored = sorted(((jaccard(qsh, doc_sh[i]), CORPUS[i]) for i in lsh_candidates(query)),
                    key=lambda x: -x[0])
    return [d for _, d in scored[:k]]

# k-NN brute-force (barcha hujjatlar)
def retrieve_bruteforce(query, k=5):
    qsh = shingles(query)
    scored = sorted(((jaccard(qsh, doc_sh[i]), CORPUS[i]) for i in range(len(CORPUS))),
                    key=lambda x: -x[0])
    return [d for _, d in scored[:k]]

assert cand is not None and len(cand) > 0, "LSH nomzodlari bo'sh. lsh_candidates ni tekshiring."
assert len(cand) < len(CORPUS), "LSH nomzodlari jami hujjatdan kam bo'lishi kerak (tezlik!)."
assert retrieve_lsh(query, 1)[0] == retrieve_bruteforce(query, 1)[0], (
    "LSH eng yaqin hujjati k-NN bilan mos kelishi kerak (so'rov korpusdagi hujjat)."
)
print(f"✓ LSH {len(cand)} nomzodni tekshiradi ({len(CORPUS)} o'rniga) va eng yaqinni topadi.")


### 4D. Birgalikda: tezlikni o'lchaymiz (k-NN vs LSH)


In [ ]:
# ═══ BIRGALIKDA: tezlik taqqoslash ═══
import time
queries = CORPUS[:20]

t0 = time.perf_counter()
for q in queries: retrieve_bruteforce(q, 5)
t_bf = time.perf_counter() - t0

t0 = time.perf_counter()
for q in queries: retrieve_lsh(q, 5)
t_lsh = time.perf_counter() - t0

# === SIZNING KODINGIZ (1 qator) ===
# o'rtacha LSH nomzodlar sonini hisoblang -> avg_cand
avg_cand = None

print(f"k-NN (brute-force): {t_bf*1000:.1f} ms  ({len(CORPUS)} solishtirish/so'rov)")
print(f"LSH               : {t_lsh*1000:.1f} ms  ({avg_cand} nomzod/so'rov o'rtacha)")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert avg_cand is not None, "avg_cand None. O'rtacha nomzodlar sonini hisoblang."
assert avg_cand < len(CORPUS), (
    f"O'rtacha nomzod ({avg_cand:.1f}) jami hujjatdan ({len(CORPUS)}) kam bo'lishi kerak."
)
print(f"✓ LSH har so'rovda o'rtacha {avg_cand:.1f} hujjatni tekshiradi "
      f"({len(CORPUS)} o'rniga) — kichik korpusda vaqt farqi kichik, "
      f"katta korpusda LSH ancha tez.")


### 4E. Mustaqil: O'z xato so'zingiz va so'rovingiz

Scaffold yo'q — `correct()` va `retrieve_lsh()` dan foydalaning.


In [ ]:
# ═══ MUSTAQIL (Siz qilasiz) ═══
# 1. O'z imlo xatongizni tuzating (masalan "kompyutr", "dasturr").
# 2. O'z so'rovingiz uchun eng o'xshash 3 hujjatni toping.
my_typo = "kompyutr"
# === SIZNING KODINGIZ ===
my_fixed = None
my_query = "yangi telefon"
my_results = None


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert my_fixed is not None and my_results is not None, "my_fixed/my_results None."
assert isinstance(my_fixed, str), "correct() satr qaytaradi."
assert isinstance(my_results, list) and len(my_results) <= 3, "retrieve_lsh ro'yxat (<=3)."
print(f"✓ Mustaqil: '{my_typo}' -> '{my_fixed}'; so'rov natijasi {len(my_results)} hujjat.")


---
## §5  Loyihaga Ulash — m04 SpellLSHRetriever Yozamiz (~13 daqiqa)

§4 dagi funksiyalarni rasmiy kapstone moduliga o'tkazamiz. Imzolar
`capstone/contracts.py` da belgilangan. Modul datasketch-ixtiyoriy
(toza-python MinHash+LSH) va m01 ustiga quriladi.


In [ ]:
# ═══ §5: m04 SpellLSHRetriever modulini yozamiz ═══
from pathlib import Path
REPO_ROOT = Path().resolve().parent.parent
M04_PATH = REPO_ROOT / 'capstone' / 'modules' / 'm04_spell_lsh_retriever.py'
M04_PATH.parent.mkdir(parents=True, exist_ok=True)

M04_CODE = r'''"""
capstone/modules/m04_spell_lsh_retriever.py
SpellLSHRetriever — imlo tuzatish (Noisy Channel + Levenshtein) va
MinHash LSH asosida hujjat qidirish.
Shartnoma: capstone/contracts.py :: SpellLSHRetriever
P4 (5-kun amaliyoti) da qurilgan; m01 (TextPreprocessor) ustiga quriladi.
Consumed by: m15 (agent tool: spell_correct), Day 16 (pipeline).

LSH toza-python/numpy MinHash bilan amalga oshirilgan (datasketch SHART EMAS).
Kaggle da datasketch.MinHashLSH ham ishlatilishi mumkin, lekin bu modul unga
bog'liq emas -- har joyda ishlaydi.
"""
from __future__ import annotations

import pickle
import zlib
from collections import Counter

import numpy as np

try:
    from m01_text_preprocessor import TextPreprocessor
except ImportError:   # paket sifatida import qilinganda
    from .m01_text_preprocessor import TextPreprocessor


def _stable_hash(s: str) -> int:
    """Takrorlanuvchan (deterministik) token xeshi -- Python hash() tuzsiz."""
    return zlib.crc32(s.encode("utf-8")) & 0x7FFFFFFF


class SpellLSHRetriever:
    """Imlo tuzatish (Noisy Channel) + LSH asosida hujjat qidirish.

    Consumed by: m15 (agent tool: spell_correct), Day 16 (pipeline).
    """

    def __init__(self, num_perm: int = 64, bands: int = 16,
                 alpha_channel: float = 0.1) -> None:
        self._pre = TextPreprocessor()
        self._num_perm = num_perm
        self._bands = bands
        self._rows = num_perm // bands
        self._alpha = alpha_channel
        rng = np.random.RandomState(42)
        self._a = rng.randint(1, 2**31 - 1, num_perm).astype(np.int64)
        self._b = rng.randint(0, 2**31 - 1, num_perm).astype(np.int64)
        self._p = np.int64(2**31 - 1)
        self._freq: Counter = Counter()
        self._total = 0
        self._docs: list[str] = []
        self._doc_shingles: list[set] = []
        self._buckets: dict = {}

    # ─── imlo: Levenshtein + Noisy Channel ──────────────────────────────────────
    def edit_distance(self, s1: str, s2: str) -> int:
        """Levenshtein tahrir masofasi (dinamik dasturlash, 2D jadval)."""
        m, n = len(s1), len(s2)
        D = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m + 1):
            D[i][0] = i
        for j in range(n + 1):
            D[0][j] = j
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if s1[i - 1] == s2[j - 1]:
                    D[i][j] = D[i - 1][j - 1]
                else:
                    D[i][j] = 1 + min(D[i - 1][j], D[i][j - 1], D[i - 1][j - 1])
        return D[m][n]

    def fit_dictionary(self, texts: list[str]) -> None:
        """Lug'at chastotalarini (til modeli P(w)) korpusdan o'rganadi."""
        for t in texts:
            self._freq.update(self._pre.preprocess(t))
        self._total = sum(self._freq.values())

    def correct(self, word: str) -> str:
        """Noisy channel: argmax_w P(w)*P(x|w); P(x|w)=alpha^edit_distance."""
        w = word.lower()
        if not self._total:
            return w
        if w in self._freq:        # lug'atda bor -> tuzatish shart emas
            return w
        best, best_score = w, -1.0
        for cand, c in self._freq.items():
            d = self.edit_distance(w, cand)
            if d > 2:
                continue
            score = (c / self._total) * (self._alpha ** d)
            if score > best_score:
                best, best_score = cand, score
        return best

    # ─── LSH: MinHash + banding ─────────────────────────────────────────────────
    def _shingles(self, text: str) -> set:
        return set(self._pre.preprocess(text)) if text.strip() else set()

    def _signature(self, shingles: set) -> tuple:
        if not shingles:
            return tuple([int(self._p)] * self._num_perm)
        hs = np.array([_stable_hash(t) for t in shingles], dtype=np.int64)
        M = (self._a[:, None] * hs[None, :] + self._b[:, None]) % self._p
        return tuple(int(x) for x in M.min(axis=1))

    def index_docs(self, texts: list[str]) -> None:
        """Hujjatlarni MinHash LSH indeksiga qo'shadi."""
        for text in texts:
            i = len(self._docs)
            sh = self._shingles(text)
            self._docs.append(text)
            self._doc_shingles.append(sh)
            sig = self._signature(sh)
            for band in range(self._bands):
                key = (band, sig[band * self._rows:(band + 1) * self._rows])
                self._buckets.setdefault(key, set()).add(i)

    @staticmethod
    def _jaccard(a: set, b: set) -> float:
        if not a or not b:
            return 0.0
        return len(a & b) / len(a | b)

    def lsh_candidates(self, query: str) -> set:
        """LSH savatlaridan nomzod hujjat indekslarini qaytaradi (tezlik uchun)."""
        sig = self._signature(self._shingles(query))
        cand = set()
        for band in range(self._bands):
            key = (band, sig[band * self._rows:(band + 1) * self._rows])
            cand |= self._buckets.get(key, set())
        return cand

    def retrieve_lsh(self, query: str, k: int = 5) -> list[str]:
        """LSH orqali eng o'xshash k ta hujjatni qaytaradi."""
        qsh = self._shingles(query)
        scored = [
            (self._jaccard(qsh, self._doc_shingles[i]), self._docs[i])
            for i in self.lsh_candidates(query)
        ]
        scored.sort(key=lambda x: -x[0])
        return [doc for _, doc in scored[:k]]

    # ─── saqlash / yuklash ──────────────────────────────────────────────────────
    def save(self, path: str) -> None:
        state = {
            "num_perm": self._num_perm, "bands": self._bands, "rows": self._rows,
            "alpha": self._alpha, "a": self._a, "b": self._b,
            "freq": self._freq, "total": self._total,
            "docs": self._docs, "doc_shingles": self._doc_shingles,
            "buckets": self._buckets,
        }
        with open(path, "wb") as f:
            pickle.dump(state, f)

    def load(self, path: str) -> None:
        with open(path, "rb") as f:
            s = pickle.load(f)
        self._pre = TextPreprocessor()
        self._num_perm, self._bands, self._rows = s["num_perm"], s["bands"], s["rows"]
        self._alpha, self._a, self._b = s["alpha"], s["a"], s["b"]
        self._p = np.int64(2**31 - 1)
        self._freq, self._total = s["freq"], s["total"]
        self._docs, self._doc_shingles = s["docs"], s["doc_shingles"]
        self._buckets = s["buckets"]
'''

M04_PATH.write_text(M04_CODE, encoding='utf-8')
print(f'OK: {M04_PATH} yozildi ({len(M04_CODE)} belgi).')


In [ ]:
# ─── m04 ni import qilib, shartnomaga muvofiqligini tekshiramiz ──────────────
import sys, importlib, os, tempfile
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))
import m04_spell_lsh_retriever
importlib.reload(m04_spell_lsh_retriever)
from m04_spell_lsh_retriever import SpellLSHRetriever

sp = SpellLSHRetriever()

# Shartnoma: edit_distance -> int (L4 [I3] qulflangan qiymat)
assert sp.edit_distance("qo'l", "ko'l") == 1          # Ma'ruza L4 [I3]-slayd
assert sp.edit_distance("dastur", "dastir") == 1

# Shartnoma: correct -> str  (lug'atni o'rgatamiz)
sp.fit_dictionary(CORPUS)
assert sp.correct("telfon") == "telefon"
assert sp.correct("internt") == "internet"

# Shartnoma: index_docs / retrieve_lsh -> list[str]
sp.index_docs(CORPUS)
res = sp.retrieve_lsh(CORPUS[0], k=3)
assert isinstance(res, list) and len(res) >= 1 and isinstance(res[0], str)
assert res[0] == CORPUS[0]    # so'rov korpus hujjati -> o'zi eng yaqin

# Shartnoma: save/load roundtrip
tmp = os.path.join(tempfile.gettempdir(), "m04_test.pkl")
sp.save(tmp)
sp2 = SpellLSHRetriever(); sp2.load(tmp)
assert sp2.correct("telfon") == "telefon"
assert sp2.retrieve_lsh(CORPUS[0], 1)[0] == CORPUS[0]

_ed = sp.edit_distance("qo'l", "ko'l")
_cor = sp.correct("telfon")
print("✓ m04 SpellLSHRetriever barcha shartnoma tekshiruvlaridan o'tdi!")
print(f"  edit_distance(qo'l, ko'l)={_ed}; correct('telfon')={_cor}")


### 5C. Git — m04 ni Commit Qiling

```bash
git add capstone/modules/m04_spell_lsh_retriever.py
git commit -m "day05: capstone - m04 SpellLSHRetriever"
git push origin HEAD
```


In [ ]:
import subprocess
res = subprocess.run(["git", "add", "capstone/modules/m04_spell_lsh_retriever.py"],
                     capture_output=True, text=True, cwd=str(REPO_ROOT))
print("git add:", "OK" if res.returncode == 0 else res.stderr.strip())
st = subprocess.run(["git", "diff", "--cached", "--name-only"],
                    capture_output=True, text=True, cwd=str(REPO_ROOT))
if st.stdout.strip():
    cm = subprocess.run(["git", "commit", "-m", "day05: capstone - m04 SpellLSHRetriever"],
                        capture_output=True, text=True, cwd=str(REPO_ROOT))
    print(cm.stdout.strip() or "Commit bajarildi.")
else:
    print("Commit qilish uchun yangi o'zgarish yo'q.")


---
## §6  Tadqiqot Savoli + Yakun (~7 daqiqa)

**Savol:** O'zbek imlosida apostrof variantlari (`o'` / `oʻ` / `o`) va
agglutinatsiya tuzatishni qiyinlashtiradi. Nega? LSH "taxminiy" — qachon
o'xshash hujjatni o'tkazib yuborishi mumkin?


In [ ]:
# Mini tadqiqot: apostrof va agglutinatsiya tuzatishga ta'siri
print("Apostrof variantlari orasidagi tahrir masofasi:")
for x, y in [("qo'l", "qol"), ("to'g'ri", "togri"), ("ma'no", "mano")]:
    print(f"  edit_distance({x!r}, {y!r}) = {edit_distance(x, y)}")

print("\nAgglutinatsiya — bitta o'zak, ko'p nomzod:")
forms = ["telefon", "telefonni", "telefonga", "telefonlar"]
print(f"  '{forms[0]}' atrofidagi shakllar: {forms}")
print(f"  edit_distance('telefon','telefonlar') = {edit_distance('telefon','telefonlar')}")
print("  -> nomzodlar fazosi kattalashadi; normallashtirish (m01) yordam beradi.")

print("\nLSH taxminiyligi: band/qator sozlamasi false-negative berishi mumkin")
print("(o'xshash hujjat boshqa savatga tushib, topilmay qolishi).")


---
## Yakun

**Bugun nimalar qildik:**
- ✓ Levenshtein tahrir masofasini DP bilan yozdik (L4 [I3]: edit('qo'l','ko'l')=1)
- ✓ Noisy Channel imlo tuzatish: telfon->telefon, internt->internet
- ✓ MinHash LSH indeks qurdik va k-NN bilan tezlikni taqqosladik
- ✓ m04 `SpellLSHRetriever` ni capstone moduliga yozdik (datasketch-ixtiyoriy)

**1-hafta yakuni (w1 milestone):** m01 -> m02 -> m03 -> m04 ni birlashtirib,
hujjat yordamchingizning klassik yadrosini sinab ko'ramiz.

---

> **Chiqish chiptasi:** Bugun eng tushunarsiz qolgan narsa nima?
> (Quyidagi katakka yozing.)
